<a href="https://colab.research.google.com/github/Boberineo/finance-assistant/blob/main/spending_analyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Spending Analyser

This will use a month of bank transactions and track where the money went with totals and percent for each category, then the program will be able to give gemini powered money coach questions about the results.

## How this assignment is organized

**The six-step method:**

Planning (before any code)

*   Steps 1–4: problem, inputs/outputs, example, pseudocode

Step 5: Convert it to Python

*   Setup
*   Loading and checking the CSV
*   Custom tool: category totals and percentages
*   The Gemini finance assistant
*   The Gradio interface

Step 6: Test with a variety of data

*   Tests, bug log, second data file



### Step 1: The Problem

-The program will be for a uni student living on campus who also works a part time job. They have just ran out of money last month just before pay day and don't understand why. They have gotten a CSV export but that raw data is not very insightful for them.

-This person would beifit from knowing more insightful information on the CSV like 'where did my money actually go?'. Weather they spent more than they earned this month. Someone to ask follow up qyestions who answers using their data.

-So the main objective is to return a monthly spending rebiew that rurns raw transactions into a category breakdown and personalised advise.



## Step 2: Identify the inputs and outputs

Inputs:

-Transactions file. Format: Columns date, category, amount, description. Spending is negative, income is positive.

-Questions. Format: Plain text, e.g. "What should I cut back on?"

-API key: Format: Never typed into the notebook but it stays in colab secrets.

Outputs:

-Total spent, total income, money left.

-Table of spending per category

-Friendly message if the file is wrong

-Assistant's answer

## Step 3: Example

A small example of some transactions in a month:

date, category, amount, description

2026-09-01 / Income / 450.00 / Café wages

2026-09-02 / Groceries / -32.50 / Campus shop

2026-09-03 / Takeaway	/ -18.00	/ Pizza

2026-09-04	/ Transport	/ -12.00	/ Bus pass top-up

2026-09-06	/ Groceries	/ -21.50	/ Big shop

2026-09-07	/ Takeaway	/ -9.00	/ Chips


1. Ignore positive amounts for the categories. They are income. Income = **$450.00**.

2. Add up spending per category:
   - Groceries: 32.50 + 21.50 = **$54.00**
   - Takeaway: 18.00 + 9.00 = **$27.00**
   - Transport: **$12.00**
3. Total spent = 54 + 27 + 12 = **$93.00**
4. Percentage of spending = category ÷ total x 100, rounded to 1 decimal place:
   - Groceries: 54 ÷ 93 x 100 = 58.06…  **58.1%**
   - Takeaway: 27 ÷ 93 x 100 = 29.03…  **29.0%**
   - Transport: 12 ÷ 93 x 100 = 12.90…  **12.9%**
   - Check: 58.1 + 29.0 + 12.9 = 100.0
5. Money left = income - spending = 450 - 93 = **$357.00**

These exact numbers become the ones used in the testing section.

## Step 4: Pseudocode

**Load and check the file**

FUNCTION load_transactions(file_path):

IF no file given -> return nothing + "Please upload a CSV file"

IF file name doesn't end in .csv -> return nothing + "That isn't a CSV file"

TRY to read the file with pandas

IF the file can't be found or is empty -> return nothing + friendly message

tidy the column names (lower case, no spaces around them)

IF any of date, category, amount, description is missing -> return nothing + "missing columns #"

turn amount into numbers; anything that isn't a number becomes blank (NaN)

count and remove rows where amount is blank (keep rows that only miss a category)

IF no rows left -> return nothing + message

return the table + "Loaded N transactions (skipped M bad rows)"

**Custom tool: category totals and percentages**

FUNCTION category_summary(transactions):

IF transactions is not a list -> return nothing + error message

totals = empty dictionary

total_spent = 0, total_income = 0

FOR each row in transactions:

amount = row's amount as a float

IF amount isn't a number -> skip it

IF amount > 0 -> add to total_income

ELIF amount < 0:

spend = amount without the minus sign

category = tidy text (strip spaces, Title Case, blank -> "Uncategorised")

totals[category] = totals.get(category, 0) + spend

total_spent = total_spent + spend

breakdown = empty list

WHILE totals still has categories:

biggest = the category with the largest total (max)

IF total_spent is 0 -> percent = 0    (guard against divide-by-zero)

ELSE percent = totals[biggest] / total_spent * 100, rounded to 1 dp

add {category, total, percent} to breakdown

delete biggest from totals

return total_spent, total_income, money_left, and the breakdown

**Assistant**

FUNCTION build_prompt(summary, question, chat_so_far):

write the summary out as text lines ("Groceries: $54.00 (58.1%)")

IF there's a conversation so far -> add it, so follow-up questions make sense

add the user's question

return the whole text

FUNCTION ask_gemini(prompt):
IF no API key -> return "assistant is offline" message

TRY main model

IF error is 503 (busy) -> TRY fallback model

IF error is 429 / 401 / 403 -> say what it means and what to do

IF anything else fails -> return friendly error, don't crash

**Interface**

file upload at the top (used by both tabs)

TAB "Where your money went": Analyse button -> load_transactions -> category_summary

-> show status, totals, category table

TAB "Ask Penny": question box + example questions + Ask button

-> build_prompt -> ask_gemini -> add to the conversation box

IF the user asks before uploading, or asks nothing -> show a hint, don't call Gemini

"Clear chat" button empties the conversation

##Step 5: Convert it to Python
Setup

Install the two libraries, then import everything in one place.

pandas: reads the CSV into a table.

gradio: builds the web interface.

google-genai: Google's official library for talking to Gemini.

math: used once, to spot "not a number" values.

In [2]:
!pip install -q gradio google-genai

In [3]:
import math
import pandas as pd

# Folder that holds the sample CSV files that come with this project
DATA_FOLDER = "data"

REPO_URL = "https://github.com/Boberineo/finance-assistant"

try:
    with open(DATA_FOLDER + "/transactions_september.csv") as f:
        pass
except FileNotFoundError:
    !git clone -q {REPO_URL}
    DATA_FOLDER = "finance-assistant/data"

print("Using data from:", DATA_FOLDER)

fatal: destination path 'finance-assistant' already exists and is not an empty directory.
Using data from: finance-assistant/data


###Loading and checking the CSV

load_transactions follows the pseudocode from step 4. It always gives back two things, the table and a message for the user. Every error message says what went wrong and what to do next.

In [4]:
REQUIRED_COLUMNS = ["date", "category", "amount", "description"]


def load_transactions(file_path):
    """Read a transactions CSV and check it.

    Returns (table, message). If the file can't be used, table is None and
    message explains why in plain English.
    """
    # 1. Was a file given at all?
    if file_path is None or file_path == "":
        return None, "Please upload a CSV file first."

    # 2. Is it a CSV? (.lower() so that 'DATA.CSV' also counts)
    if not file_path.lower().endswith(".csv"):
        return None, "That file isn't a CSV. Please upload a .csv file exported from your bank."

    # 3. Try to read it. Specific problems first, then a general catch-all.
    try:
        table = pd.read_csv(file_path)
    except FileNotFoundError:
        return None, "I couldn't find that file. Please upload it again."
    except pd.errors.EmptyDataError:
        return None, "That file is empty: there's nothing in it to analyse."
    except pd.errors.ParserError as error:
        return None, ("One row in your file has too many commas, which usually means an amount "
                      "was written like 1,000.00. Change it to 1000.00 and upload the file again. "
                      f"(Details: {str(error).strip()})")
    except Exception as error:
        return None, f"I couldn't read that file as a CSV ({error}). Please check it opens in Excel or Sheets."

    # 4. Tidy the column names: ' Amount ' -> 'amount'
    clean_names = []
    for name in table.columns:
        clean_names.append(str(name).strip().lower())
    table.columns = clean_names

    # 5. Are all the columns I need there?
    missing = ""
    for column in REQUIRED_COLUMNS:
        if column not in table.columns:
            missing += column + " "
    if missing != "":
        return None, ("This doesn't look like a transactions file. Missing column(s): "
                      + missing.strip() + ". The file needs: date, category, amount, description.")

    # 6. Clean the amounts, then make them numbers (the cleaning pattern from the course).
    table["amount"] = table["amount"].astype(str).str.replace("$", "", regex=False)
    table["amount"] = table["amount"].str.replace(",", "", regex=False)
    table["amount"] = pd.to_numeric(table["amount"], errors="coerce")

    # 7. Drop rows with no usable amount, and count how many went.
    rows_before = len(table)
    table = table.dropna(subset=["amount"])
    bad_rows = rows_before - len(table)

    # 8. Anything left?
    if table.empty:
        return None, "There are no usable transactions in that file. Check the amount column has numbers in it."

    message = f"Loaded {len(table)} transactions."
    if bad_rows > 0:
        message += f" Skipped {bad_rows} row(s) where the amount wasn't a number."
    return table, message

Quick check: load the September file and look at the first few rows.

In [5]:
september, message = load_transactions(DATA_FOLDER + "/transactions_september.csv")
print(message)
september.head()

Loaded 37 transactions.


,date,category,amount,description
0,2026-09-01,Income,310.00,Cafe wages (fortnight 1)
1,2026-09-01,Subscriptions,-10.99,Spotify
2,2026-09-02,Groceries,-24.60,Campus shop
3,2026-09-02,Takeaway,-14.50,Domino's
4,2026-09-03,Transport,-3.00,Bus into town


### Custom tool: category totals and % of spending

This is the main part of what the app does and it is from my own logic. It takes a plain Python list of dictionaries so it doesn't depend on pandas at all, and I can test it with tiny hand made lists.



In [6]:
def category_summary(transactions):
    """Work out total spending per category and each category's % of all spending.

    transactions: a list of dictionaries, each with 'category' and 'amount'.
    Spending is negative, income is positive.
    Returns (summary, message). summary is None if the input can't be used.
    """
    # Bad input: the tool needs a list (e.g. someone passes a number or a string)
    if type(transactions) != list:
        return None, "The category tool needs a list of transactions."

    totals = {}
    total_spent = 0
    total_income = 0
    rows_used = 0

    for row in transactions:
        # Skip anything that isn't a dictionary with an amount in it
        if type(row) != dict or "amount" not in row:
            continue

        # Turn the amount into a number; skip the row if that isn't possible
        try:
            amount = float(row["amount"])
        except ValueError:
            continue
        except TypeError:
            continue
        if math.isnan(amount):
            continue

        rows_used += 1

        if amount > 0:
            total_income += amount
        elif amount < 0:
            spend = -amount
            # Tidy the name so 'takeaway ', 'Takeaway' and 'TAKEAWAY' are one category.
            category = str(row.get("category", "")).strip().title()
            if category == "" or category == "Nan":
                category = "Uncategorised"
            totals[category] = totals.get(category, 0) + spend
            total_spent += spend

    if rows_used == 0:
        return None, "None of those transactions had an amount I could use."

    # Build the breakdown biggest first: find the biggest category, add it to the list,
    # remove it from the dictionary, and repeat until the dictionary is empty.
    breakdown = []
    while len(totals) > 0:
        biggest = max(totals, key=totals.get)
        percent = round(totals[biggest] / total_spent * 100, 1)
        breakdown.append({"category": biggest, "total": round(totals[biggest], 2), "percent": percent})
        del totals[biggest]

    summary = {
        "total_spent": round(total_spent, 2),
        "total_income": round(total_income, 2),
        "money_left": round(total_income - total_spent, 2),
        "transactions_used": rows_used,
        "categories": breakdown,
    }
    if total_spent == 0:
        return summary, "No spending found, only income."
    return summary, f"Summarised {rows_used} transactions across {len(breakdown)} spending categories."

In [7]:
hand_example = [
    {"date": "2026-09-01", "category": "Income",    "amount": 450.00, "description": "Café wages"},
    {"date": "2026-09-02", "category": "Groceries", "amount": -32.50, "description": "Campus shop"},
    {"date": "2026-09-03", "category": "Takeaway",  "amount": -18.00, "description": "Pizza"},
    {"date": "2026-09-04", "category": "Transport", "amount": -12.00, "description": "Bus pass top-up"},
    {"date": "2026-09-06", "category": "Groceries", "amount": -21.50, "description": "Big shop"},
    {"date": "2026-09-07", "category": "Takeaway",  "amount": -9.00,  "description": "Chips"},
]
summary, message = category_summary(hand_example)
print(message)
print(summary)

Summarised 6 transactions across 3 spending categories.
{'total_spent': 93.0, 'total_income': 450.0, 'money_left': 357.0, 'transactions_used': 6, 'categories': [{'category': 'Groceries', 'total': 54.0, 'percent': 58.1}, {'category': 'Takeaway', 'total': 27.0, 'percent': 29.0}, {'category': 'Transport', 'total': 12.0, 'percent': 12.9}]}


The loader gives a pandas table, but the tool wants a list of dictionaries. table_to_rows turns the two columns it needs into ordinary lists, then builds one dictionary per row.

In [8]:
def table_to_rows(table):
    """Turn the pandas table into a list of {'category': ..., 'amount': ...} dictionaries."""
    categories = list(table["category"])
    amounts = list(table["amount"])
    rows = []
    for i in range(len(amounts)):
        rows.append({"category": categories[i], "amount": amounts[i]})
    return rows

###Displaying the Result

In [9]:
def dollars(amount):
    """Format a number as money: 12.5 -> '$12.50', -23.36 -> '-$23.36'."""
    if amount < 0:
        return f"-${-amount:.2f}"
    return f"${amount:.2f}"


def totals_text(summary):
    """Three lines: total spent, total income, money left."""
    left = dollars(summary["money_left"])
    if summary["money_left"] < 0:
        left += " (you spent more than came in)"
    return (f"Total spent:  {dollars(summary['total_spent'])}\n"
            f"Total income: {dollars(summary['total_income'])}\n"
            f"Money left:   {left}")


def summary_to_rows(summary):
    """The category breakdown as a list of rows for the interface's table."""
    rows = []
    for item in summary["categories"]:
        rows.append([item["category"], dollars(item["total"]), f"{item['percent']:.1f}%"])
    return rows


def summary_to_text(summary):
    """The whole summary as plain lines (this is what Gemini will be given)."""
    if summary is None:
        return "No data uploaded."
    text = totals_text(summary) + "\nSpending by category (biggest first):\n"
    for item in summary["categories"]:
        text += f"- {item['category']}: {dollars(item['total'])} ({item['percent']:.1f}% of spending)\n"
    return text

In [10]:
september, message = load_transactions(DATA_FOLDER + "/transactions_september.csv")
september_summary, message = category_summary(table_to_rows(september))
print(message)
print(summary_to_text(september_summary))

Summarised 37 transactions across 8 spending categories.
Total spent:  $678.36
Total income: $655.00
Money left:   -$23.36 (you spent more than came in)
Spending by category (biggest first):
- Going Out: $216.00 (31.8% of spending)
- Shopping: $175.00 (25.8% of spending)
- Takeaway: $121.40 (17.9% of spending)
- Groceries: $94.00 (13.9% of spending)
- Books: $24.99 (3.7% of spending)
- Subscriptions: $22.97 (3.4% of spending)
- Transport: $12.00 (1.8% of spending)
- Phone: $12.00 (1.8% of spending)



The Gemini finance assistant
Four parts, kept separate so each can be checked on its own:

SYSTEM_INSTRUCTION sets the persona and the rules.

build_prompt puts the user's own summary, the chat so far and their question into one piece of text.

ask_gemini sends it to the model.

check_question / answer_question catch blank, too-long or too-early questions before anything is sent.

In [11]:
from google import genai
from google.genai import types

MAIN_MODEL = "gemini-flash-latest"
FALLBACK_MODEL = "gemini-3.5-flash-lite"

# Get the key from Colab Secrets. Outside Colab (or if the secret is missing) there's
# no key, and the app runs in "offline" mode instead of crashing.
try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = None

if API_KEY:
    client = genai.Client(api_key=API_KEY)
    print("Gemini key found: assistant is online.")
else:
    client = None
    print("No GOOGLE_API_KEY found: the assistant will reply with an 'offline' message.")

Gemini key found: assistant is online.


In [12]:
SYSTEM_INSTRUCTION = """
You are Penny, a friendly and practical money coach for Australian university students.

Your user is a student living on campus with a part-time job, who ran out of money
before payday last month. Each message gives you a summary of THEIR transactions for
the month, worked out by the app, followed by their question.

Rules:
- Base every answer on the figures in the summary. Quote their real $ amounts and
  percentages. Never invent numbers or transactions that aren't in the summary.
- If something can't be answered from the summary (e.g. which day they ran out),
  say what the data doesn't show instead of guessing.
- Give specific, realistic suggestions a student could actually do this month,
  with rough $ savings where you can work them out from their figures.
- Keep answers short: under 150 words, plain English, no jargon, no lecturing.
- If the question is off-topic (not about money, budgeting or their spending),
  say kindly in one sentence that you can only help with their spending, and
  suggest a money question they could ask instead.
- If the question is unclear, ask one short clarifying question.
- Don't recommend specific investments, crypto, loans or credit cards. If they
  mention debt or real hardship, suggest their university's student support or
  financial counselling service.
"""

In [13]:
def build_prompt(summary, question, chat_so_far=""):
    """Put the user's data, the chat so far and the new question into one prompt."""
    prompt = "MY SPENDING THIS MONTH (calculated from my uploaded transactions):\n"
    prompt += summary_to_text(summary) + "\n"

    # Earlier questions and answers, so Penny can follow up ("what about takeaways?")
    if chat_so_far is not None and chat_so_far.strip() != "":
        prompt += "OUR CONVERSATION SO FAR:\n" + chat_so_far.strip() + "\n\n"

    prompt += "MY QUESTION: " + question.strip()
    return prompt


def ask_gemini(prompt):
    """Send the prompt to Gemini with the persona. Always returns text, never crashes."""
    if client is None:
        return ("(Assistant offline: no GOOGLE_API_KEY found. Add it in Colab Secrets "
                "and re-run the setup cell. Your spending summary still works.)")

    settings = types.GenerateContentConfig(system_instruction=SYSTEM_INSTRUCTION)

    for model_name in [MAIN_MODEL, FALLBACK_MODEL]:
        try:
            response = client.models.generate_content(
                model=model_name, contents=prompt, config=settings)
            if response.text:
                return response.text
            return "Sorry, I didn't get an answer back. Could you rephrase the question?"
        except Exception as error:
            details = str(error)
            if "503" in details and model_name == MAIN_MODEL:
                print("503 from", MAIN_MODEL, "- trying", FALLBACK_MODEL)
                continue
            elif "503" in details:
                return "Gemini is very busy right now (503). Please wait a minute and try again."
            elif "429" in details:
                return "Too many questions in a short time (429). Please wait a minute and try again."
            elif "401" in details or "403" in details or "API_KEY_INVALID" in details:
                return "Gemini didn't accept the API key. Check GOOGLE_API_KEY in Colab Secrets."
            else:
                return f"Sorry, I couldn't get an answer from Gemini ({details})."

    return "Gemini is very busy right now (503). Please wait a minute and try again."


def check_question(question, summary):
    """Return a hint if the question can't be sent yet, or "" if it's fine."""
    if question is None or question.strip() == "":
        return "Type a question about your spending first, e.g. 'What should I cut back on?'"
    if summary is None:
        return "Upload your transactions CSV first, so I can see your numbers."
    if len(question) > 1000:
        return "That's a very long message. Could you ask it in a sentence or two?"
    return ""


def answer_question(question, summary, chat_so_far=""):
    """Check the question, then ask Gemini. Returns the reply text."""
    problem = check_question(question, summary)
    if problem != "":
        return problem
    return ask_gemini(build_prompt(summary, question, chat_so_far))

###What Gemini receives

In [14]:
october, message = load_transactions(DATA_FOLDER + "/transactions_october.csv")
october_summary, message = category_summary(table_to_rows(october))

question = "Why did I run out of money, and what should I cut first?"
print("===== SEPTEMBER =====")
print(build_prompt(september_summary, question))
print("\n===== OCTOBER =====")
print(build_prompt(october_summary, question))

===== SEPTEMBER =====
MY SPENDING THIS MONTH (calculated from my uploaded transactions):
Total spent:  $678.36
Total income: $655.00
Money left:   -$23.36 (you spent more than came in)
Spending by category (biggest first):
- Going Out: $216.00 (31.8% of spending)
- Shopping: $175.00 (25.8% of spending)
- Takeaway: $121.40 (17.9% of spending)
- Groceries: $94.00 (13.9% of spending)
- Books: $24.99 (3.7% of spending)
- Subscriptions: $22.97 (3.4% of spending)
- Transport: $12.00 (1.8% of spending)
- Phone: $12.00 (1.8% of spending)

MY QUESTION: Why did I run out of money, and what should I cut first?

===== OCTOBER =====
MY SPENDING THIS MONTH (calculated from my uploaded transactions):
Total spent:  $394.19
Total income: $650.00
Money left:   $255.81
Spending by category (biggest first):
- Groceries: $185.40 (47.0% of spending)
- Laptop Repair: $89.00 (22.6% of spending)
- Going Out: $54.00 (13.7% of spending)
- Takeaway: $22.30 (5.7% of spending)
- Transport: $12.00 (3.0% of spending)

In [15]:
# Ask Gemini (needs the API key)
print(answer_question(question, september_summary))

You ran out because you spent $678.36, which was $23.36 more than your $655.00 income. 

Your biggest expenses were **Going Out** ($216.00, or 31.8%) and **Shopping** ($175.00, or 25.8%). You also spent $121.40 (17.9%) on **Takeaway**—more than your entire $94.00 grocery bill!

Here is what I'd cut first:
- **Halve your takeaway:** Cooking campus meals instead of ordering out could easily save you around $60 of that $121.40.
- **Trim your nights out:** Trimming your $216.00 social spend by just 25% (like having one less drink or finding free campus events) saves roughly $54.

Together, those two small tweaks would free up around $114 this month—instantly wiping out your $23.36 deficit and leaving you cash to spare before payday.


### The Gradio interface


The interface only calls the functions above. All the calculating stays in them, so it can be tested without the interface.

Layout:
* File upload at the top, shared by both tabs.
* Tab 1, "Where your money went":
* Tab 2, "Ask Penny":


In [16]:
import gradio as gr


def analyse_file(file_path):
    """Analyse button: load the file, run the custom tool, start a fresh chat.

    Returns (status, totals, table rows, conversation).
    """
    table, message = load_transactions(file_path)
    if table is None:
        return "⚠️ " + message, "", [], ""

    summary, tool_message = category_summary(table_to_rows(table))
    if summary is None:
        return "⚠️ " + tool_message, "", [], ""

    return "✅ " + message + " " + tool_message, totals_text(summary), summary_to_rows(summary), ""


def ask_penny(question, file_path, chat_text):
    """Ask button: answer using the uploaded file, and add the exchange to the conversation.

    Returns (conversation, question box, hint).
    """
    if chat_text is None:
        chat_text = ""

    # Work the summary out again from the uploaded file, so the answer always
    # matches the file that's on screen right now
    summary = None
    if file_path is not None:
        table, message = load_transactions(file_path)
        if table is not None:
            summary, tool_message = category_summary(table_to_rows(table))

    problem = check_question(question, summary)
    if problem != "":
        return chat_text, question, "⚠️ " + problem

    reply = ask_gemini(build_prompt(summary, question, chat_text))
    chat_text += "You: " + question.strip() + "\n" + "Penny: " + reply + "\n\n"
    return chat_text, "", ""


def clear_chat():
    """Clear chat button: empty the conversation and the hint."""
    return "", ""


with gr.Blocks(title="Student Spending Analyser") as demo:
    gr.Markdown("# 💵 Student Spending Analyser\n"
                "Upload a month of transactions from your bank, see where your money went, "
                "then ask Penny, your money coach.")

    file_input = gr.File(label="Your transactions CSV", file_types=[".csv"])

    with gr.Tab("1. Where your money went"):
        analyse_button = gr.Button("Analyse my spending", variant="primary")
        status_box = gr.Textbox(label="Status", value="Upload a file, then press Analyse.",
                                interactive=False)
        totals_box = gr.Textbox(label="Totals", lines=3, interactive=False)
        table_box = gr.Dataframe(headers=["Category", "Spent", "% of spending"],
                                 label="Spending by category (biggest first)", interactive=False)

    with gr.Tab("2. Ask Penny"):
        question_box = gr.Textbox(label="Your question",
                                  placeholder="e.g. Why did I run out of money before payday?",
                                  info="Penny answers using the numbers from your uploaded file.")
        gr.Examples(examples=[["Why did I run out of money before payday?"],
                              ["What should I cut back on first?"],
                              ["How much would I save if I halved my takeaways?"]],
                    inputs=[question_box])
        ask_button = gr.Button("Ask", variant="primary")
        hint_box = gr.Textbox(label="Message", interactive=False)
        chat_box = gr.Textbox(label="Conversation", lines=14, interactive=False)
        clear_button = gr.Button("Clear chat")

    # Connect the buttons to the functions: inputs go in, outputs come back in the same order
    analyse_button.click(fn=analyse_file, inputs=[file_input],
                         outputs=[status_box, totals_box, table_box, chat_box])
    # Also analyse straight away when a file is uploaded or removed, so an old summary
    # never stays on screen next to a different file (found while testing)
    file_input.change(fn=analyse_file, inputs=[file_input],
                      outputs=[status_box, totals_box, table_box, chat_box])
    ask_button.click(fn=ask_penny, inputs=[question_box, file_input, chat_box],
                     outputs=[chat_box, question_box, hint_box])
    clear_button.click(fn=clear_chat, outputs=[chat_box, hint_box])

In [17]:
# share=True gives a public link, so the app opens in a new browser tab from Colab
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://197ad05f3961d6fe3d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 6: Test with a variety of data

Each test is a small function full of asserts that check specific expected values. The asserts act as a specification of what the code must do. If one is false, Python raises an AssertionError. The runner at the bottom catches it, so one failing test doesn't stop the rest, and prints a pass/fail list.

In [18]:
# ---------- Loading the CSV ----------

def test_load_september():
    table, message = load_transactions(DATA_FOLDER + "/transactions_september.csv")
    assert len(table) == 37
    assert message == "Loaded 37 transactions."


def test_load_no_file():
    for nothing in [None, ""]:
        table, message = load_transactions(nothing)
        assert table is None
        assert "upload" in message.lower()


def test_load_not_csv():
    table, message = load_transactions("my_budget.xlsx")
    assert table is None
    assert "isn't a CSV" in message


def test_load_missing_file():
    table, message = load_transactions(DATA_FOLDER + "/does_not_exist.csv")
    assert table is None
    assert "couldn't find" in message


def test_load_empty_file():
    table, message = load_transactions(DATA_FOLDER + "/empty.csv")
    assert table is None
    assert "empty" in message


def test_load_wrong_columns():
    table, message = load_transactions(DATA_FOLDER + "/wrong_columns.csv")
    assert table is None
    assert "Missing column(s): date category amount description" in message


def test_load_broken_row():
    table, message = load_transactions(DATA_FOLDER + "/broken_row.csv")
    assert table is None
    assert "too many commas" in message
    assert "1000.00" in message


def test_load_bad_amounts():
    table, message = load_transactions(DATA_FOLDER + "/bad_amounts.csv")
    assert len(table) == 4
    assert -5.0 in list(table["amount"])
    assert "Skipped 2 row(s) where the amount wasn't a number" in message


# ---------- Custom tool ----------

def test_tool_hand_example():   # the numbers I worked out by hand in step 3
    summary, message = category_summary(hand_example)
    assert summary["total_spent"] == 93.00
    assert summary["total_income"] == 450.00
    assert summary["money_left"] == 357.00
    assert summary["categories"] == [
        {"category": "Groceries", "total": 54.00, "percent": 58.1},
        {"category": "Takeaway",  "total": 27.00, "percent": 29.0},
        {"category": "Transport", "total": 12.00, "percent": 12.9},
    ]


def test_tool_september_file():
    summary, message = category_summary(table_to_rows(september))
    assert summary["total_spent"] == 678.36
    assert summary["total_income"] == 655.00
    assert summary["money_left"] == -23.36
    assert summary["categories"][0] == {"category": "Going Out", "total": 216.00, "percent": 31.8}


def test_tool_merges_spellings_in_real_file():
    # September has 'takeaway ' and 'groceries' rows; they must join the main categories
    names = []
    for item in september_summary["categories"]:
        names.append(item["category"])
    assert "Takeaway" in names and "takeaway " not in names and "Takeaway " not in names
    assert "Groceries" in names and "groceries" not in names
    assert september_summary["categories"][2] == {"category": "Takeaway", "total": 121.40, "percent": 17.9}


def test_tool_october_file_is_different():
    summary, message = category_summary(table_to_rows(october))
    assert summary["money_left"] == 255.81
    assert summary["categories"][0]["category"] == "Groceries"
    assert summary != september_summary


def test_tool_agrees_with_pandas():
    # October's category names are already tidy, so groupby can be compared directly.
    spending = october[october["amount"] < 0]
    pandas_totals = spending.groupby("category")["amount"].sum()
    for item in october_summary["categories"]:
        assert abs(-pandas_totals[item["category"]] - item["total"]) < 0.01, item["category"]
    assert len(pandas_totals) == len(october_summary["categories"])


def test_tool_percentages_add_up():
    for summary in [september_summary, october_summary]:
        total_percent = 0
        for item in summary["categories"]:
            total_percent += item["percent"]
        assert abs(total_percent - 100) <= 0.5, total_percent


def test_tool_biggest_first():
    categories = september_summary["categories"]
    for i in range(1, len(categories)):
        assert categories[i]["total"] <= categories[i - 1]["total"]


def test_tool_only_income():  # boundary: no spending at all, so nothing to divide by
    summary, message = category_summary([{"category": "Income", "amount": 200}])
    assert summary["total_spent"] == 0
    assert summary["money_left"] == 200
    assert summary["categories"] == []
    assert message == "No spending found, only income."


def test_tool_merges_category_spellings():
    rows = [{"category": "Food", "amount": -5},
            {"category": "food ", "amount": -3},
            {"category": "FOOD", "amount": -2}]
    summary, message = category_summary(rows)
    assert summary["categories"] == [{"category": "Food", "total": 10.0, "percent": 100.0}]


def test_tool_blank_category():
    rows = [{"category": "", "amount": -4}, {"category": float("nan"), "amount": -6}]
    summary, message = category_summary(rows)
    assert summary["categories"] == [{"category": "Uncategorised", "total": 10.0, "percent": 100.0}]


def test_tool_skips_bad_rows():
    rows = [{"category": "Food", "amount": -10},
            {"category": "Food", "amount": "twelve"},
            {"category": "Food", "amount": None},
            {"category": "Food", "amount": float("nan")},
            {"category": "Food"},
            "not a dictionary",
            {"category": "Food", "amount": "-2.5"},
            {"category": "Bus", "amount": 0}]
    summary, message = category_summary(rows)
    assert summary["total_spent"] == 12.5
    assert summary["transactions_used"] == 3
    assert len(summary["categories"]) == 1


def test_tool_wrong_input_type():
    for bad in [None, 42, "transactions.csv", {"amount": -5}]:
        summary, message = category_summary(bad)
        assert summary is None
        assert "needs a list" in message


def test_tool_empty_list():
    summary, message = category_summary([])
    assert summary is None
    assert "None of those transactions" in message


def test_table_to_rows():
    rows = table_to_rows(october)
    assert len(rows) == 22
    assert rows[0] == {"category": "Income", "amount": 320.0}


def test_dollars_format():
    assert dollars(12.5) == "$12.50"
    assert dollars(-23.36) == "-$23.36"
    assert dollars(0) == "$0.00"


# ---------- Assistant (no live Gemini calls) ----------

def test_prompt_uses_users_data():
    sept_prompt = build_prompt(september_summary, "What should I cut?")
    oct_prompt = build_prompt(october_summary, "What should I cut?")
    assert "Going Out: $216.00 (31.8% of spending)" in sept_prompt
    assert "-$23.36" in sept_prompt
    assert "$255.81" in oct_prompt and "$216.00" not in oct_prompt
    assert sept_prompt != oct_prompt
    assert sept_prompt.endswith("MY QUESTION: What should I cut?")


def test_prompt_includes_conversation():
    chat = "You: Why am I short?\nPenny: You spent $23.36 more than came in.\n\n"
    prompt = build_prompt(september_summary, "What about takeaways?", chat)
    assert "OUR CONVERSATION SO FAR:\nYou: Why am I short?" in prompt
    assert "OUR CONVERSATION SO FAR" not in build_prompt(september_summary, "Hi", "")


def test_check_question():   # these stop the question before Gemini is called
    assert "Type a question" in check_question("   ", september_summary)
    assert "Type a question" in check_question(None, september_summary)
    assert "Upload your transactions" in check_question("Hi", None)
    assert "very long message" in check_question("a" * 1001, september_summary)
    assert check_question("What should I cut?", september_summary) == ""


def test_offline_message():
    if client is None:          # only meaningful when there's no API key
        assert "Assistant offline" in ask_gemini("anything")


# ---------- Interface functions ----------

def test_interface_bad_file():
    status, totals, rows, chat = analyse_file(DATA_FOLDER + "/wrong_columns.csv")
    assert status.startswith("⚠️")
    assert totals == "" and rows == []


def test_interface_good_file():
    status, totals, rows, chat = analyse_file(DATA_FOLDER + "/transactions_october.csv")
    assert status.startswith("✅")
    assert "Money left:   $255.81" in totals
    assert rows[0] == ["Groceries", "$185.40", "47.0%"]


def test_interface_question_before_upload():
    chat, question_box, hint = ask_penny("What should I cut?", None, "")
    assert chat == ""                                   # nothing added to the conversation
    assert "Upload your transactions CSV" in hint
    assert question_box == "What should I cut?"         # their question isn't wiped


def test_interface_blank_question():
    chat, question_box, hint = ask_penny("  ", DATA_FOLDER + "/transactions_september.csv", "")
    assert chat == "" and "Type a question" in hint

In [19]:
# Run every test and report. Each entry is "name": function (the function itself, not called yet).
all_tests = {
    "load september": test_load_september,
    "load: no file": test_load_no_file,
    "load: not a CSV": test_load_not_csv,
    "load: missing file": test_load_missing_file,
    "load: empty file": test_load_empty_file,
    "load: wrong columns": test_load_wrong_columns,
    "load: broken row 🐞": test_load_broken_row,
    "load: bad amounts 🐞": test_load_bad_amounts,
    "tool: hand example (step 3)": test_tool_hand_example,
    "tool: september file": test_tool_september_file,
    "tool: merges spellings in real file": test_tool_merges_spellings_in_real_file,
    "tool: october is different (R2)": test_tool_october_file_is_different,
    "tool: agrees with pandas": test_tool_agrees_with_pandas,
    "tool: percentages add up": test_tool_percentages_add_up,
    "tool: biggest first": test_tool_biggest_first,
    "tool: only income": test_tool_only_income,
    "tool: merges category spellings": test_tool_merges_category_spellings,
    "tool: blank category": test_tool_blank_category,
    "tool: skips bad rows": test_tool_skips_bad_rows,
    "tool: wrong input type": test_tool_wrong_input_type,
    "tool: empty list": test_tool_empty_list,
    "table_to_rows": test_table_to_rows,
    "dollars format": test_dollars_format,
    "prompt uses user's data (R2)": test_prompt_uses_users_data,
    "prompt includes conversation": test_prompt_includes_conversation,
    "check_question": test_check_question,
    "offline message": test_offline_message,
    "interface: bad file": test_interface_bad_file,
    "interface: good file": test_interface_good_file,
    "interface: question before upload": test_interface_question_before_upload,
    "interface: blank question": test_interface_blank_question,
}

passed = 0
for name, test in all_tests.items():
    try:
        test()
        print("PASS ", name)
        passed += 1
    except AssertionError as error:
        print("FAIL ", name, error)
    except Exception as error:
        print("ERROR", name, error)

print(f"\n{passed} of {len(all_tests)} tests passed")

PASS  load september
PASS  load: no file
PASS  load: not a CSV
PASS  load: missing file
PASS  load: empty file
PASS  load: wrong columns
PASS  load: broken row 🐞
PASS  load: bad amounts 🐞
PASS  tool: hand example (step 3)
PASS  tool: september file
PASS  tool: merges spellings in real file
PASS  tool: october is different (R2)
PASS  tool: agrees with pandas
PASS  tool: percentages add up
PASS  tool: biggest first
PASS  tool: only income
PASS  tool: merges category spellings
PASS  tool: blank category
PASS  tool: skips bad rows
PASS  tool: wrong input type
PASS  tool: empty list
PASS  table_to_rows
PASS  dollars format
PASS  prompt uses user's data (R2)
PASS  prompt includes conversation
PASS  check_question
PASS  offline message
PASS  interface: bad file
PASS  interface: good file
PASS  interface: question before upload
PASS  interface: blank question

31 of 31 tests passed
